## Index embeddings into embedding store

## Set up

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [103]:
import os
import sys

import torch
from dotenv import load_dotenv
from loguru import logger
from pydantic import BaseModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

import mlflow
from sqlalchemy import create_engine
import pandas as pd
from loguru import logger

sys.path.insert(0, "..")
_ = load_dotenv(override = True)

In [75]:
class Args(BaseModel):
    testing: bool = False
    run_name: str = "000-first-attempt"
    notebook_persist_dp: str = None
    random_seed: int = 41
    device: str = None

    top_K: int = 100
    top_k: int = 10

    embedding_dim: int = 128

    mlf_model_name: str = "item2vec"

    batch_recs_fp: str = None

    qdrant_url: str = None
    qdrant_collection_name: str = None

    oltp_url: str = None
    article_metadata_table: str = "article_metadata"
    oltp_user: str = None
    oltp_password: str = None
    oltp_db: str = None



    def init(self):
        self.notebook_persist_dp = os.path.abspath(f"data/{self.run_name}")
        os.makedirs(self.notebook_persist_dp, exist_ok=True)
        self.batch_recs_fp = f"{self.notebook_persist_dp}/batch_recs.jsonl"

        if not (qdrant_host := os.getenv("QDRANT_HOST")):
            raise Exception(f"Environment variable QDRANT_HOST is not set.")

        qdrant_port = os.getenv("QDRANT_PORT")
        self.qdrant_url = f"{qdrant_host}:{qdrant_port}"
        self.qdrant_collection_name = os.getenv("QDRANT_COLLECTION_NAME")

        self.oltp_url = os.getenv("POSTGRES_HOST") + ":" + os.getenv("POSTGRES_PORT")
        self.oltp_user = os.getenv("POSTGRES_USER")
        self.oltp_password = os.getenv("POSTGRES_PASSWORD")
        self.oltp_db = os.getenv("POSTGRES_DB")
        return self


args = Args().init()

print(args.model_dump_json(indent=2))

{
  "testing": false,
  "run_name": "000-first-attempt",
  "notebook_persist_dp": "/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/000-first-attempt",
  "random_seed": 41,
  "device": null,
  "top_K": 100,
  "top_k": 10,
  "embedding_dim": 128,
  "mlf_model_name": "item2vec",
  "batch_recs_fp": "/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/000-first-attempt/batch_recs.jsonl",
  "qdrant_url": "localhost:6333",
  "qdrant_collection_name": "item2vec",
  "oltp_url": "localhost:5432",
  "article_metadata_table": "article_metadata",
  "oltp_user": "lastfirstkiss",
  "oltp_password": "nightchange",
  "oltp_db": "hm-recsys"
}


### Load MLflow model

In [11]:
mlf_client = mlflow.MlflowClient()

In [12]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{args.mlf_model_name}@champion")

In [13]:
run_id = model.metadata.run_id
run_info = mlf_client.get_run(run_id).info
artifact_uri = run_info.artifact_uri

In [18]:
sample_input = mlflow.artifacts.load_dict(f"{artifact_uri}/inferrer/input_example.json")
sample_input

{'item_1_ids': ['153115020'], 'item_2_ids': ['160442007']}

In [19]:
prediction = model.predict(sample_input)
prediction

{'item_1_ids': ['153115020'],
 'item_2_ids': ['160442007'],
 'scores': [0.4289277195930481]}

In [22]:
model

mlflow.pyfunc.loaded_model:
  artifact_path: inferrer
  flavor: mlflow.pyfunc.model
  run_id: d9ffa207c9e24227898a8fa189de7db0

### Get embeddings

In [25]:
# Reference:https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html#mlflow.pyfunc.PyFuncModel.unwrap_python_model

In [27]:
skipgram_model = model.unwrap_python_model().model
embedding_0 = skipgram_model.embeddings(torch.tensor(0))
embedding_dim = embedding_0.size()[0]
embedding_dim

128

In [49]:
full_embeddings = skipgram_model.embeddings.weight.detach().cpu().numpy()

In [50]:
full_embeddings

array([[ 1.3207403e-01, -6.4548127e-02, -2.6660672e-01, ...,
         2.8644717e-01,  2.7122268e-01,  1.4914775e-01],
       [-1.7728357e-02, -8.9434855e-02,  1.5192871e-01, ...,
         3.0962429e-03,  2.4663532e-01, -4.6155748e-01],
       [ 5.1990741e-01, -1.2096022e+00,  6.1852378e-01, ...,
        -2.8858805e-01, -2.1766253e-01, -7.3434019e-01],
       ...,
       [ 3.3491412e-01, -7.4059236e-01,  5.3838706e-01, ...,
        -8.2463533e-01,  3.2758173e-02, -1.0248208e-01],
       [ 2.9488653e-01,  2.9372209e-01, -5.6379765e-01, ...,
        -2.1193759e-03, -1.3788483e-02, -7.6268017e-01],
       [-6.1193022e-40,  5.0115338e-40, -6.1307789e-40, ...,
         4.9089587e-40, -5.4838414e-40,  5.5562885e-40]], dtype=float32)

In [52]:
full_embeddings.shape

(646, 128)

In [62]:
idm = model.unwrap_python_model().id_mapping

In [65]:
idm["index_to_item"][:4]

['153115020', '160442007', '160442010', '179950001']

In [78]:
DATABASE_URL = f"postgresql://{args.oltp_user}:{args.oltp_password}@{args.oltp_url}/{args.oltp_db}"

# Create an engine
engine = create_engine(DATABASE_URL)

# Read table into Pandas DataFrame
table_name = args.article_metadata_table
article_metadata_df = pd.read_sql(f"SELECT * FROM oltp.{table_name}", engine)

article_metadata_df.head(3)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,220094001,220094,Aguilera maxidress,265,Dress,Garment Full body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Strapless maxi dress in jersey with an elastic...
1,220094010,220094,Aguilera maxidress,265,Dress,Garment Full body,1010016,Solid,19,Greenish Khaki,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Strapless maxi dress in jersey with an elastic...
2,220094011,220094,Aguilera maxidress,265,Dress,Garment Full body,1010017,Stripe,73,Dark Blue,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Strapless maxi dress in jersey with an elastic...


In [82]:
article_metadata_df.loc[article_metadata_df["article_id"] == 220094001].drop(columns=["article_id"]).to_dict(orient="records")[0]


{'product_code': 220094,
 'prod_name': 'Aguilera maxidress',
 'product_type_no': 265,
 'product_type_name': 'Dress',
 'product_group_name': 'Garment Full body',
 'graphical_appearance_no': 1010016,
 'graphical_appearance_name': 'Solid',
 'colour_group_code': 9,
 'colour_group_name': 'Black',
 'perceived_colour_value_id': 4,
 'perceived_colour_value_name': 'Dark',
 'perceived_colour_master_id': 5,
 'perceived_colour_master_name': 'Black',
 'department_no': 1676,
 'department_name': 'Jersey Basic',
 'index_code': 'A',
 'index_name': 'Ladieswear',
 'index_group_no': 1,
 'index_group_name': 'Ladieswear',
 'section_no': 16,
 'section_name': 'Womens Everyday Basics',
 'garment_group_no': 1002,
 'garment_group_name': 'Jersey Basic',
 'detail_desc': 'Strapless maxi dress in jersey with an elasticated seam at the waist and slits in the sides. Integral top with elastication at the top.'}

In [104]:
def get_metadata(article_idx: int, idm: dict):
    try:
        article_id = int(idm["index_to_item"][article_idx])
        metadata = article_metadata_df.loc[article_metadata_df["article_id"] == article_id].drop(columns=["article_id"]).to_dict(orient="records")[0]
        return metadata
    except:
        logger.warning(f"Article index {article_idx} not found in metadata. Return None")
        return None

In [105]:
get_metadata(2, idm)

{'product_code': 160442,
 'prod_name': '3p Sneaker Socks',
 'product_type_no': 302,
 'product_type_name': 'Socks',
 'product_group_name': 'Socks & Tights',
 'graphical_appearance_no': 1010016,
 'graphical_appearance_name': 'Solid',
 'colour_group_code': 10,
 'colour_group_name': 'White',
 'perceived_colour_value_id': 3,
 'perceived_colour_value_name': 'Light',
 'perceived_colour_master_id': 9,
 'perceived_colour_master_name': 'White',
 'department_no': 3611,
 'department_name': 'Shopbasket Socks',
 'index_code': 'B',
 'index_name': 'Lingeries/Tights',
 'index_group_no': 1,
 'index_group_name': 'Ladieswear',
 'section_no': 62,
 'section_name': 'Womens Nightwear, Socks & Tigh',
 'garment_group_no': 1021,
 'garment_group_name': 'Socks and Tights',
 'detail_desc': 'Short, fine-knit socks designed to be hidden by your shoes with a silicone trim at the back of the heel to keep them in place.'}

In [106]:
# Create payload for each embedding and save to a list
article_metadata_list = []
# Get rid of the last embedding as it is the padding embedding
for i in range(full_embeddings.shape[0] ) :
    payload = get_metadata(i, idm)
    article_metadata_list.append(payload)

2025-02-04 15:49:43.424 | WARNING  | __main__:get_metadata:7 - Article index 645 not found in metadata. Return None


## Embedding store

In [107]:
ann_index = QdrantClient(url=args.qdrant_url)

In [55]:
collection_exists = ann_index.collection_exists(args.qdrant_collection_name)
if collection_exists:
    logger.info(f"Deleting existing Qdrant collection {args.qdrant_collection_name}...")
    ann_index.delete_collection(args.qdrant_collection_name)

create_collection_result = ann_index.create_collection(
    collection_name=args.qdrant_collection_name,
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
)

assert create_collection_result == True

In [108]:
upsert_result = ann_index.upsert(
    collection_name=args.qdrant_collection_name,
    points=[
        PointStruct(id=idx, vector=vector.tolist(), payload=article_metadata_list[idx])
        for idx, vector in enumerate(full_embeddings)
    ],
)
assert str(upsert_result.status) == "completed"
upsert_result

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [112]:
hits = ann_index.search(
    collection_name=args.qdrant_collection_name,
    query_vector=full_embeddings[2],
    limit=args.top_K,
)

/tmp/ipykernel_172008/1877292219.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = ann_index.search(


In [113]:
hits

[ScoredPoint(id=2, version=0, score=1.0000001, payload={'product_code': 160442, 'prod_name': '3p Sneaker Socks', 'product_type_no': 302, 'product_type_name': 'Socks', 'product_group_name': 'Socks & Tights', 'graphical_appearance_no': 1010016, 'graphical_appearance_name': 'Solid', 'colour_group_code': 10, 'colour_group_name': 'White', 'perceived_colour_value_id': 3, 'perceived_colour_value_name': 'Light', 'perceived_colour_master_id': 9, 'perceived_colour_master_name': 'White', 'department_no': 3611, 'department_name': 'Shopbasket Socks', 'index_code': 'B', 'index_name': 'Lingeries/Tights', 'index_group_no': 1, 'index_group_name': 'Ladieswear', 'section_no': 62, 'section_name': 'Womens Nightwear, Socks & Tigh', 'garment_group_no': 1021, 'garment_group_name': 'Socks and Tights', 'detail_desc': 'Short, fine-knit socks designed to be hidden by your shoes with a silicone trim at the back of the heel to keep them in place.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=14